<a href="https://colab.research.google.com/github/metalBigLesson/PythonChess/blob/main/Chess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
class Piece:
    def __init__(self, game, char, x=-1, y=-1, color_sign=2):
        self.game = game
        self.char = char
        self.x = x
        self.y = y
        self.color_sign = color_sign #white: 1, black: -1
        self.char = char
        self.moved = False

    def move(self, x, y):
        self.game.board[self.y][self.x] = Empty(self.game, self.x, self.y)
        self.game.board[y][x] = self
        self.x = x
        self.y = y
        self.moved = True

    def __str__(self):
        if (self.color_sign == -1):
            return chr(ord(self.char) + 6)
        else:
            return self.char

    def __mul__(self, other):
        return self.color_sign * other.color_sign

    def is_empty(self): return self.color_sign == 0

    def can_move(self, x, y): return False

class Empty(Piece):
    def __init__(self, game, x=-1, y=-1):
        super().__init__(game, ' ', x, y, 0)

    def __str__(self):
        return '□' if (self.x + self.y) % 2 == 1 else '■'

class Enpassant(Piece):
    def __init__(self, game, x, y):
        super().__init__(game, '◐', x, y, 0)

class Pawn(Piece):
    def __init__(self, game, x=-1, y=-1, color_sign=2):
        super().__init__(game, '♙', x, y, color_sign)

    def can_move(self, x, y):
        x_diff = x - self.x
        y_diff_forward = (y - self.y) * self.color_sign
        if x_diff == 0 and y_diff_forward == 1:
            return self.game.get_piece(x, y).is_empty() #move
        if abs(x_diff) == 1 and y_diff_forward == 1:
            if isinstance(self.game.get_piece(x, y), Enpassant): #enpassant
                self.game.special_move = Special_Move.Enpassant
                return True
            return self.game.get_piece(x, y) * self == -1 #take-move
        if x_diff == 0 and y_diff_forward == 2: #first move
            if self.moved: return False
            self.game.special_move = Special_Move.Pre_Enpassant
            return self.game.get_piece(x, y).is_empty()
        return False

class Knight(Piece):
    def __init__(self, game, x=-1, y=-1, color_sign=2):
        super().__init__(game, '♘', x, y, color_sign)

    def can_move(self, x, y):
        x_diff = x - self.x
        y_diff = y - self.y
        return x_diff * x_diff + y_diff * y_diff == 5 and self * self.game.get_piece(x, y) != 1

class Bishop(Piece):
    def __init__(self, game, x=-1, y=-1, color_sign=2):
        super().__init__(game, '♗', x, y, color_sign)

    def can_move(self, x, y):
        x_diff = x - self.x
        y_diff = y - self.y
        if x_diff == 0 or y_diff == 0: return False
        if self.game.get_piece(x, y) * self == 1: return False
        slope = y_diff / x_diff
        if (slope == 1):
            j = min(self.y, y) + 1
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if not self.game.get_piece(i, j).is_empty(): return False
                j += 1
            return True
        if (slope == -1):
            j = max(self.y, y) - 1
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if not self.game.get_piece(i, j).is_empty(): return False
                j -= 1
            return True
        return False

class Rook(Piece):

    def __init__(self, game, x=-1, y=-1, color_sign=2):
        super().__init__(game, '♖', x, y, color_sign)

    def can_move(self, x, y):
        if (x == self.x and y == self.y): return False
        if self.game.get_piece(x, y) * self == 1: return False
        if (x == self.x):
            for i in range(min(self.y, y) + 1, max(self.y, y)):
                if not self.game.get_piece(x, i).is_empty(): return False
            return True
        if (y == self.y):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if not self.game.get_piece(i, y).is_empty(): return False
            return True
        return False

class Queen(Piece):

    def __init__(self, game, x=-1, y=-1, color_sign=2):
        super().__init__(game, '♕', x, y, color_sign)

    def can_move(self, x, y):
        temp_rook = Rook(self.game, self.x, self.y, self.color_sign)
        temp_bishop = Bishop(self.game, self.x, self.y, self.color_sign)
        return temp_rook.can_move(x, y) or temp_bishop.can_move(x, y)

class King(Piece):

    def __init__(self, game, x=-1, y=-1, color_sign=2):
        super().__init__(game, '♔', x, y, color_sign)

    def can_move(self, x, y):
        x_diff = x - self.x
        y_diff = y - self.y
        if abs(x_diff) <= 1 and abs(y_diff) <= 1: return self.game.get_piece(x, y) * self != 1
        if self.moved or self.game.is_check(): return False
        if x_diff == 2 and y_diff == 0:
            if self.game.get_piece(7, self.y).moved or self.game.is_check(5): return False
            for i in range(5, 7):
                if not self.game.get_piece(i, self.y).is_empty(): return False
            self.game.special_move = Special_Move.Castling
            return True
        if x_diff == -2 and y_diff == 0:
            if self.game.get_piece(0, self.y).moved or self.game.is_check(3): return False
            for i in range(1, 4):
                if not self.game.get_piece(i, self.y).is_empty(): return False
            self.game.special_move = Special_Move.Castling
            return True
        return False

In [7]:
from enum import Enum
class Special_Move(Enum):
    Normal = 0
    Pre_Enpassant = 1
    Enpassant = 2
    Castling = 3

In [8]:
from enum import Enum
import copy

def rank(rank, color_sign):
    if color_sign == 1:
        return rank - 1
    else:
        return 8 - rank

class Game:
    def __init__(self):
        self.special_move = Special_Move.Normal
        self.turn_color_sign = -1
        self.board = []
        self.kings = [King(self), King(self)]
        self.selected_piece = None
        for y in range(8):
            self.board.append([])
            for x in range(8):
                match y:
                    case 1: self.board[y].append(Pawn(self, x, y, 1))
                    case 6: self.board[y].append(Pawn(self, x, y, -1))
                    case _: self.board[y].append(Empty(self, x, y))
        self._place_rank(0, 1, [Rook(self), Knight(self), Bishop(self), Queen(self), self.kings[0], Bishop(self), Knight(self), Rook(self)])
        self._place_rank(7, -1, [Rook(self), Knight(self), Bishop(self), Queen(self), self.kings[1], Bishop(self), Knight(self), Rook(self)])
        self.end_turn()

    def click(self, x, y):
        if self.selected_piece is None: #선택
            piece = self.get_piece(x, y)
            if piece.color_sign != self.turn_color_sign:
                print('잘못된 선택.')
                return
            self.selected_piece = piece
            print('선택된 기물(', piece, ') 이동: ')
            return
        if self.selected_piece.x == x and self.selected_piece.y == y: #선택 취소
            self.selected_piece = None
            print('선택 취소.')
            return
        if not self.selected_piece.can_move(x, y):
            print('잘못된 이동.')
            return
        self.move_piece(self.selected_piece, x, y)

    def move_piece(self, piece, x, y):
        board_backup = copy.deepcopy(self.board)
        kings_backup = copy.deepcopy(self.kings)
        if (self.special_move == Special_Move.Enpassant): #앙파상
            y_rear = y - self.turn_color_sign
            self.board[y_rear][x] = Empty(self, x, y_rear)
        piece.move(x, y)
        if (self.special_move == Special_Move.Pre_Enpassant): #앙파상전
            y_rear = y - game.turn_color_sign
            self.board[y_rear][x] = Enpassant(game, x, y_rear)
        elif (self.special_move == Special_Move.Castling): #캐슬링
            if x == 6:
                self.get_piece(7, y).move(5, y)
            else:
                self.get_piece(0, y).move(3, y)
        if (self.is_check()):#자가 체크
            print('자가 체크')
            self.board = board_backup
            self.kings = kings_backup
            return
        if isinstance(piece, Pawn) and y == rank(8, self.turn_color_sign):
            while True:
                promotion = input('프로모션: (q, r, b, n)')
                match promotion:
                    case 'q': self.board[y][x] = Queen(self, x, y, game.turn_color_sign)
                    case 'r': self.board[y][x] = Rook(self, x, y, game.turn_color_sign)
                    case 'b': self.board[y][x] = Bishop(self, x, y, game.turn_color_sign)
                    case 'n': self.board[y][x] = Knight(self, x, y, game.turn_color_sign)
                    case _: continue
                break
        self.end_turn()

    def king(self, color_sign):
        return self.kings[0] if color_sign == 1 else self.kings[1]

    def _place_rank(self, y, color_sign, pieces):
        for x in range(len(pieces)):
            piece = pieces[x]
            piece.x = x
            piece.y = y
            piece.color_sign = color_sign
            self.board[y][x] = piece

    def get_piece(self, x, y):
        return self.board[y][x]

    def is_check(self, x=-1, y=-1):
        if x == -1: x = self.king(self.turn_color_sign).x
        if y == -1: y = self.king(self.turn_color_sign).y
        for i in range(8):
            for piece in self.board[i]:
                if piece.color_sign == self.turn_color_sign or isinstance(piece, King): continue
                if piece.can_move(x, y):
                    self.special_move = Special_Move.Normal
                    return True
        self.special_move = Special_Move.Normal
        return False

    def end_turn(self):
        self.turn_color_sign *= -1
        self.selected_piece = None
        print('  a b c d e f g h')
        for y in range(7, -1, -1):
            print(y + 1, end=' ')
            for x in range(8):
                piece = self.get_piece(x, y)
                char = str(self.get_piece(x, y))
                if char == '◐' and y == rank(3, self.turn_color_sign):
                    self.board[y][x] = Empty(self, x, y)
                    char = str(Empty(self, x, y))
                print(char, end='')
            print('', y + 1)
        print('  a b c d e f g h')
        print('백' if self.turn_color_sign == 1 else '흑', '기물 선택: ')

In [9]:
game = Game()
while True:
    input_string = input()
    if len(input_string) != 2:
        print('잘못된 입력')
        continue
    x = ord(input_string[0]) - ord('a')
    y = int(input_string[1]) - 1
    if x >= 0 and x <= 7 and y >= 0 and y <= 7: game.click(x, y)

  a b c d e f g h
8 ♜♞♝♛♚♝♞♜ 8
7 ♟♟♟♟♟♟♟♟ 7
6 □■□■□■□■ 6
5 ■□■□■□■□ 5
4 □■□■□■□■ 4
3 ■□■□■□■□ 3
2 ♙♙♙♙♙♙♙♙ 2
1 ♖♘♗♕♔♗♘♖ 1
  a b c d e f g h
백 기물 선택: 
e1
선택된 기물( ♔ ) 이동: 
e2
잘못된 이동.
e1
선택 취소.
e2
선택된 기물( ♙ ) 이동: 
e4
  a b c d e f g h
8 ♜♞♝♛♚♝♞♜ 8
7 ♟♟♟♟♟♟♟♟ 7
6 □■□■□■□■ 6
5 ■□■□■□■□ 5
4 □■□■♙■□■ 4
3 ■□■□◐□■□ 3
2 ♙♙♙♙□♙♙♙ 2
1 ♖♘♗♕♔♗♘♖ 1
  a b c d e f g h
흑 기물 선택: 


KeyboardInterrupt: Interrupted by user

1. 턴 시작

2. 기물 좌표 입력 -(오류발생)-> 1로

3. 자신의 기물인가? -거짓-> 1로

4. 이동할 좌표 입력 -(오류발생)-> 1로

5. 이동 가능한가? -거짓-> 1로

6. 앙파상인가? -거짓-> 8로

7. 이동

8. 앙파상 조건을 만드는가? -> 앙파상용 기물 생성

10. 캐슬링인가? -> 룩 이동

11. 자가 체크인가? -참-> 이동 취소 -> 1로

12. 프로모션인가? -참-> 프로모션

13. 턴 종료 -> 1로

폰이 2칸 전진하면 뒤에 투명 기물을 추가해서 앙파상 하게 하자